In [68]:
import glob, os
from pathlib import Path

### Move manually outputted geojson to correct dir

In [44]:
from pathlib import Path
from tqdm.auto import tqdm
import shutil, os

src_dir = Path('/home/dayn/QuPath/bin')            # where the *.geojson are
dst_root = Path('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice')

fns = sorted(src_dir.glob('*.geojson'))

for fn in tqdm(fns):
    mouse_n = fn.stem                 # 'mouse_1', 'mouse_2', ...
    dest_dir = dst_root / mouse_n / 'qu_path'   # or 'qu_path/arx' if you prefer
    dest_dir.mkdir(parents=True, exist_ok=True)

    dest_file = dest_dir / fn.name    # <-- full file path, not just the directory

    try:
        # use copy (no xattrs) to avoid EACCES on network shares
        shutil.copy(str(fn), str(dest_file))
    except PermissionError:
        # last-ditch fallback: manual copy
        with open(fn, 'rb') as r, open(dest_file, 'wb') as w:
            w.write(r.read())

print("done")


  0%|          | 0/10 [00:00<?, ?it/s]

done


# Convert geojson to zarr labels

In [110]:
from pathlib import Path
import json
from tqdm.auto import tqdm
import numpy as np
import zarr
import dask.array as da
from affine import Affine
from shapely.geometry import shape
from shapely.validation import make_valid
from rasterio import features

base = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice")

for mouse_dir in tqdm(sorted(base.glob("mouse_*")), total=len(sorted(base.glob("mouse_*")))):
    zarr_dir = mouse_dir / "zarr"
    geojson_dir = mouse_dir / "qu_path"

    if not zarr_dir.exists() or not geojson_dir.exists():
        continue

    # Zarrs to process (skip .prev / .jnotebook)
    zarrs = [
        z for z in zarr_dir.glob("*.zarr")
        if not any(s in z.name for s in [".prev", ".jnotebook"])
    ]
    geojsons = list(geojson_dir.glob("*.geojson"))
    if not geojsons:
        continue

    for zarr_fn in zarrs:
        print(f"\n🧩 Processing {zarr_fn.name}")

        # open zarr
        root = zarr.open(str(zarr_fn), mode="a")

        # use level 0/0 as main image
        arr = da.from_zarr(f"{zarr_fn}/0/0")   # (..., Y, X)
        shape_yx = arr.shape[-2:]

        # identity pixel transform (row, col) -> (y, x)
        transform = Affine.identity()

        # one merged mask per Zarr
        merged_mask = np.zeros(shape_yx, dtype="uint8")

        for gj in geojsons:
            print(f"  importing {gj.name}")
            with open(gj) as f:
                gj_data = json.load(f)

            geoms = []
            for feat in gj_data.get("features", []):
                geom = feat.get("geometry")
                if geom:
                    g = shape(geom)
                    if g.area > 2000:  # skip huge polygons (likely FOV contour)
                        continue
                    # fix invalids/self-intersections
                    try:
                        g = make_valid(g)
                    except Exception:
                        g = g.buffer(0)
                    if not g.is_empty:
                        geoms.append(g)

            if not geoms:
                print("   ⚠️ no geometries found, skipping")
                continue

            # --- drop the largest polygon (likely the FOV contour) ---
            if len(geoms) > 1:
                areas = [g.area for g in geoms]
                max_idx = int(np.argmax(areas))
                geoms = [g for i, g in enumerate(geoms) if i != max_idx]

            if not geoms:
                print("   ⚠️ only FOV contour present, skipping")
                continue

            # burn polygons -> binary mask
            mask = features.rasterize(
                [(g, 1) for g in geoms],
                out_shape=shape_yx,
                fill=0,
                dtype="uint8",
                transform=transform,
            )

            # OR into the merged mask
            merged_mask |= mask

        # ---- write as NGFF label: labels/ground_truth_mtb/0 ----
        labels_root = root.require_group("labels")
        lbl_group = labels_root.require_group("ground_truth_mtb")

        # overwrite dataset '0' if it exists
        if "0" in lbl_group:
            del lbl_group["0"]

        arr_out = lbl_group.create_array(
            "0",
            shape=merged_mask.shape,
            dtype="uint8",
            chunks="auto",
        )
        arr_out[...] = merged_mask

        # minimal NGFF label metadata
        lbl_group.attrs["multiscales"] = [{
            "name": "ground_truth_mtb",
            "version": "0.4",
            "axes": [
                {"name": "y", "type": "space", "unit": "pixel"},
                {"name": "x", "type": "space", "unit": "pixel"},
            ],
            "datasets": [{"path": "0"}],
        }]
        lbl_group.attrs["image-label"] = {
            "version": "0.4",
            "source": "..",   # parent = image root
        }

        print(f"   ✓ wrote labels/ground_truth_mtb/0 [{merged_mask.shape}]")

print("\n✅ Done — NGFF labels written as labels/ground_truth_mtb/0 in each .zarr")


  0%|          | 0/11 [00:00<?, ?it/s]


🧩 Processing 20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250814_5375.zarr
  importing mouse_1.geojson
   ✓ wrote labels/ground_truth_mtb/0 [(41702, 58291)]

🧩 Processing 20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250814_5375.zarr
  importing mouse_10.geojson
   ✓ wrote labels/ground_truth_mtb/0 [(41702, 58291)]

🧩 Processing 20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250902_5555.zarr
  importing mouse_11.geojson
   ✓ wrote labels/ground_truth_mtb/0 [(41702, 60365)]

🧩 Processing 20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5551_jnotebook.zarr
  importing mouse_2.geojson
   ✓ wrote labels/ground_truth_mtb/0 [(37555, 49997)]

🧩 Processing 20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5551.zarr
  importing mouse_2.geojson
   ✓ wrote labels/ground_truth_mtb/0 [(37555, 49997)]

🧩 Processing 20250901_40X_Time

## Fixing top/bottom pairs

